`requirements/gpu.txt`

In [1]:
import pandas as pd, gc, warnings
from qiskit_aer.noise import NoiseModel
from qiskit_aer import AerSimulator
from helpers.load_payload import load_payload
from helpers.connected_subsets import connected_subsets
from helpers.quantum_volume import quantum_volume, quantum_volume_optimised
from helpers.compare_circuits import compare_circuits
from helpers.calculate_qv import run_qv_until_fail

In [2]:
def noisy_simulator_from_payload(payload):
    '''Creates a noisy AerSimulator from the noise model and 
    coupling map in the payload.'''
    
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        noise_model = NoiseModel.from_dict(payload["noise_model"])
        
    coupling_map = payload.get("coupling_map")
    return AerSimulator(
        noise_model=noise_model,
        coupling_map=coupling_map,
        basis_gates=noise_model.basis_gates,
        method="automatic", # "statevector",
        device="GPU",
        batched_shots_gpu=True,
    )

In [8]:
def run_qv_over_subsets(backend, subsets, qv_runner):
    '''Runs quantum volume experiments over multiple subsets of qubits 
    and returns a summary of the results.'''

    ideal_backend = AerSimulator(method="statevector", device="GPU")

    summaries = []
    for i, subset in enumerate(subsets, 1):
        print(f"[{i}/{len(subsets)}] running QV on subset {subset}")

        exp_data = qv_runner(
            backend=backend,
            ideal_backend=ideal_backend,
            subset=subset,
            shots=100,
            trials=100,
            seed=42,
        )

        df = exp_data.analysis_results(dataframe=True)
        hop_row = df[df["name"] == "mean_HOP"].iloc[0]

        mean_hop = hop_row["value"].nominal_value
        hop_err  = hop_row["value"].std_dev

        summaries.append((subset, mean_hop, hop_err))

        del exp_data, df
        gc.collect()

    return summaries

In [9]:
qubits = 2 # number of qubits to run QV on
optimised = True # whether to use optimisation

calibration_path = "calibrations/ibm_marrakesh/20260129_101824.json"
payload = load_payload(calibration_path)
backend = noisy_simulator_from_payload(payload)

In [10]:
coupling_map = payload.get("coupling_map")
subsets = connected_subsets(coupling_map, qubits)

In [ ]:
expdata = run_qv_over_subsets(
    backend=backend,
    subsets=subsets,
    qv_runner=quantum_volume_optimised if optimised else quantum_volume
)

df = pd.DataFrame(expdata, columns=["subset", "mean_HOP", "hop_error"])
df = df.sort_values("mean_HOP", ascending=False).reset_index(drop=True)

out_csv = f"results/ibm_marrakesh_qv_{qubits}_tuples_{'' if not optimised else 'optimised'}.csv"
df.to_csv(out_csv, index=False)

[1/176] running QV on subset (0, 1)
[2/176] running QV on subset (1, 2)
[3/176] running QV on subset (2, 3)
[4/176] running QV on subset (3, 4)
[5/176] running QV on subset (3, 16)
[6/176] running QV on subset (4, 5)
[7/176] running QV on subset (5, 6)
[8/176] running QV on subset (6, 7)
[9/176] running QV on subset (7, 8)
[10/176] running QV on subset (7, 17)
[11/176] running QV on subset (8, 9)
[12/176] running QV on subset (9, 10)
[13/176] running QV on subset (10, 11)
[14/176] running QV on subset (11, 12)
[15/176] running QV on subset (11, 18)
[16/176] running QV on subset (12, 13)
[17/176] running QV on subset (13, 14)
[18/176] running QV on subset (14, 15)
[19/176] running QV on subset (15, 19)
[20/176] running QV on subset (16, 23)
[21/176] running QV on subset (17, 27)
[22/176] running QV on subset (18, 31)
[23/176] running QV on subset (19, 35)
[24/176] running QV on subset (20, 21)
[25/176] running QV on subset (21, 22)
[26/176] running QV on subset (21, 36)
[27/176] running

In [ ]:
ideal_backend = AerSimulator(method="statevector", device="GPU")

all_rows = []

for i, subset in enumerate(subsets, 1):
    print(f"[{i}/{len(subsets)}] subset {subset}")

    df_subset = compare_circuits(
        backend=backend,
        ideal_backend=ideal_backend,
        subset=subset,
        seed=42
    )

    df_subset["subset"] = [tuple(subset)] * len(df_subset)
    all_rows.append(df_subset)

df_all = pd.concat(all_rows, ignore_index=True)

out_csv = f"results/transpile_compare_{qubits}_tuples.csv"
df_all.drop(columns=["counts"]).to_csv(out_csv, index=False)

[1/244] subset (0, 1, 2)
[2/244] subset (1, 2, 3)
[3/244] subset (2, 3, 4)
[4/244] subset (2, 3, 16)
[5/244] subset (3, 4, 5)
[6/244] subset (3, 4, 16)
[7/244] subset (3, 16, 23)
[8/244] subset (4, 5, 6)
[9/244] subset (5, 6, 7)
[10/244] subset (6, 7, 8)
[11/244] subset (6, 7, 17)
[12/244] subset (7, 8, 9)
[13/244] subset (7, 8, 17)
[14/244] subset (7, 17, 27)
[15/244] subset (8, 9, 10)
[16/244] subset (9, 10, 11)
[17/244] subset (10, 11, 12)
[18/244] subset (10, 11, 18)
[19/244] subset (11, 12, 13)
[20/244] subset (11, 12, 18)
[21/244] subset (11, 18, 31)
[22/244] subset (12, 13, 14)
[23/244] subset (13, 14, 15)
[24/244] subset (14, 15, 19)
[25/244] subset (15, 19, 35)
[26/244] subset (16, 22, 23)
[27/244] subset (16, 23, 24)
[28/244] subset (17, 26, 27)
[29/244] subset (17, 27, 28)
[30/244] subset (18, 30, 31)
[31/244] subset (18, 31, 32)
[32/244] subset (19, 34, 35)
[33/244] subset (20, 21, 22)
[34/244] subset (20, 21, 36)
[35/244] subset (21, 22, 23)
[36/244] subset (21, 22, 36)
[3

In [4]:
df_unopt = run_qv_until_fail(
    backend=backend,
    max_n=10,
    shots=100,
    trials=100,
    seed=42,
    optimised=False
)

df_opt = run_qv_until_fail(
    backend=backend,
    max_n=10,
    shots=100,
    trials=100,
    seed=42,
    optimised=True
)

df_all = pd.concat([df_unopt, df_opt], ignore_index=True)
out_csv = "results/qv_until_fail.csv"
df_all.to_csv(out_csv, index=False)

Running QV for n = 2
Running QV for n = 3
Running QV for n = 4
Running QV for n = 5
Running QV for n = 6
Running QV for n = 7
Running QV for n = 8
Running QV for n = 9
Running QV for n = 10
Running QV for n = 2
Running QV for n = 3
Running QV for n = 4
Running QV for n = 5
Running QV for n = 6
Running QV for n = 7
Running QV for n = 8
Running QV for n = 9
Running QV for n = 10
